In [ ]:
import os
import shutil
import json
import csv
from pathlib import Path
from datetime import datetime

def reorganize_dataset(source_dir, output_dir, dry_run=False):
    source_path = Path(source_dir)
    output_path = Path(output_dir)
    
    classes = {'healthy': 'healthy', 'lesion': 'lesion', 'mrc': 'MRC'}
    views = {'cr': 'CR', 'sg': 'SG'}
    
    patient_registry = {cls: {} for cls in classes}
    patient_counter = {cls: 0 for cls in classes}
    file_mapping = {}
    patient_mapping = {}
    
    print("=" * 70)
    print("PHASE 1: Discovering patients")
    print("=" * 70)
    
    for cls, cls_display in classes.items():
        cls_path = source_path / cls
        if not cls_path.exists():
            continue
        patients_in_class = set()
        for view in views:
            view_path = cls_path / view
            if not view_path.exists():
                continue
            for filename in os.listdir(view_path):
                if filename.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')):
                    name_without_ext = os.path.splitext(filename)[0]
                    parts = name_without_ext.rsplit('-', 2)
                    patient_name = parts[0].strip().lower() if len(parts) >= 3 else name_without_ext.strip().lower()
                    patients_in_class.add(patient_name)
        
        for patient_name in sorted(patients_in_class):
            patient_counter[cls] += 1
            patient_id = f"P{patient_counter[cls]:03d}"
            patient_registry[cls][patient_name] = patient_id
            full_patient_id = f"{patient_id}_{cls_display}"
            patient_mapping[full_patient_id] = {
                'patient_id': patient_id, 'class': cls_display,
                'original_name': patient_name, 'files': []
            }
        print(f"  {cls_display}: {len(patients_in_class)} unique patients")
    
    print("\n" + "=" * 70)
    print("PHASE 2: Copying and renaming files")
    print("=" * 70)
    
    if not dry_run:
        output_path.mkdir(parents=True, exist_ok=True)
        (output_path / 'mappings').mkdir(exist_ok=True)
        for cls_display in classes.values():
            for view_display in views.values():
                (output_path / cls_display / view_display).mkdir(parents=True, exist_ok=True)
    
    total_files = 0
    for cls, cls_display in classes.items():
        cls_path = source_path / cls
        if not cls_path.exists():
            continue
        for view, view_display in views.items():
            view_path = cls_path / view
            if not view_path.exists():
                continue
            files_in_folder = 0
            for filename in sorted(os.listdir(view_path)):
                if not filename.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')):
                    continue
                original_path = view_path / filename
                name_without_ext = os.path.splitext(filename)[0]
                parts = name_without_ext.rsplit('-', 2)
                if len(parts) >= 3:
                    patient_name = parts[0].strip().lower()
                    image_num = parts[1].strip()
                else:
                    patient_name = name_without_ext.strip().lower()
                    image_num = "1"
                
                patient_id = patient_registry[cls].get(patient_name)
                if patient_id is None:
                    continue
                ext = os.path.splitext(filename)[1].lower()
                if ext == '.jpeg': ext = '.jpg'
                
                new_filename = f"{patient_id}_{cls_display}_{image_num}_{view_display}{ext}"
                new_path = output_path / cls_display / view_display / new_filename
                
                relative_original = f"{cls}/{view}/{filename}"
                file_mapping[new_filename] = relative_original
                
                full_patient_id = f"{patient_id}_{cls_display}"
                if full_patient_id in patient_mapping:
                    patient_mapping[full_patient_id]['files'].append({
                        'new_filename': new_filename,
                        'original_filename': filename, 'view': view_display
                    })
                
                if not dry_run:
                    shutil.copy2(original_path, new_path)
                files_in_folder += 1
                total_files += 1
            print(f"  {cls_display}/{view_display}: {files_in_folder} files")
    
    print(f"\n  Total files processed: {total_files}")
    
    if not dry_run:
        print("\n" + "=" * 70)
        print("PHASE 3: Saving mapping files")
        print("=" * 70)
        
        with open(output_path / 'mappings' / 'file_mapping.json', 'w', encoding='utf-8') as f:
            json.dump(file_mapping, f, indent=2, ensure_ascii=False)
        with open(output_path / 'mappings' / 'file_mapping.csv', 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['new_filename', 'original_path'])
            for new_name, orig_path in sorted(file_mapping.items()):
                writer.writerow([new_name, orig_path])
        with open(output_path / 'mappings' / 'patient_mapping.csv', 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['patient_id', 'class', 'original_name', 'num_files'])
            for full_id, info in sorted(patient_mapping.items()):
                writer.writerow([info['patient_id'], info['class'], info['original_name'], len(info['files'])])
        with open(output_path / 'mappings' / 'patient_mapping_detailed.json', 'w', encoding='utf-8') as f:
            json.dump(patient_mapping, f, indent=2, ensure_ascii=False)
        reverse_mapping = {v: k for k, v in file_mapping.items()}
        with open(output_path / 'mappings' / 'original_to_new_mapping.json', 'w', encoding='utf-8') as f:
            json.dump(reverse_mapping, f, indent=2, ensure_ascii=False)
        print("  Saved all mapping files!")
    
    print("\n" + "=" * 70)
    print("SUMMARY")
    print("=" * 70)
    print(f"  Total patients: {sum(patient_counter.values())}")
    for cls, cls_display in classes.items():
        print(f"    {cls_display}: {patient_counter[cls]} patients")
    print(f"  Total files: {total_files}")
    print(f"  Output directory: {output_path}")
    
    return file_mapping, patient_mapping

# RUN IT:
file_map, patient_map = reorganize_dataset(
    source_dir='/home/ai-hacker/projects/ODIN-Oral-Diagnostics-with-Intelligent-Networks/Data_Oct_2025',
    output_dir='/home/ai-hacker/projects/ODIN-Oral-Diagnostics-with-Intelligent-Networks/Data_Oct_2025_Data_Reorganized',
    dry_run=False  # Change to False to actually copy
)